In [ ]:
import os
import torch
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Tuple, Optional
import faiss
import pickle
import pandas as pd
import re
import easyocr

from transformers import CLIPProcessor, CLIPModel, LlamaForCausalLM, LlamaTokenizer, AutoTokenizer, AutoModelForCausalLM, AutoModelForVision2Seq, AutoProcessor
from transformers import AutoModelForVision2Seq
from typing import Optional
from qwen_vl_utils import process_vision_info



In [ ]:
import torch
torch.cuda.empty_cache()


In [ ]:
!pip install qwen-vl-utils[decord]==0.0.8

In [ ]:
from matplotlib import text


class ChemistryRAG:
    def __init__(self, 
                 train_caption_dir: str,
                 train_image_dir: str,
                 test_caption_dir: str,
                 test_image_dir: str,
                 model_path: Optional[str] = None,
                 save_dir: str = "./saved",
                 load_saved: bool = False):
        """
        Initialize the Chemistry RAG system.
        Heavy operations like building or loading indexes are moved to build().
        """
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
        self.load_saved = load_saved
        self.model_path = model_path

        # Encoder and OCR
        self.text_encoder = SentenceTransformer('BAAI/bge-large-en-v1.5')
        self.ocr = easyocr.Reader(['en'], gpu=True)

        # LLM lazy placeholders
        self.tokenizer = None
        self.llm = None
        self.processor = None

        # Dataset paths
        self.train_caption_dir = train_caption_dir
        self.train_image_dir = train_image_dir
        self.test_caption_dir = test_caption_dir
        self.test_image_dir = test_image_dir

        # These will be set later
        self.train_slides = None
        self.train_captions = None
        self.test_slides = None
        self.test_captions = None
        self.slides = None
        self.captions = None
        self.text_index = None
        self.text_embeddings = None
        self.image_index = None
        self.image_embeddings = None

    def build(self):
        """Heavy operations: load data, build or load indexes."""
        self.train_slides, self.train_captions = self._load_dataset(
            self.train_image_dir, self.train_caption_dir)
        self.test_slides, self.test_captions = self._load_dataset(
            self.test_image_dir, self.test_caption_dir)
        self.slides = self.test_slides
        self.captions = self.test_captions

        if self.load_saved and self._check_saved_files():
            print("🔵 Loading saved indexes and embeddings...")
            self._load_indexes()
        else:
            print("🟠 Building new indexes and embeddings...")
            self.text_index, self.text_embeddings = self._build_text_index()
            self.image_index, self.image_embeddings = self._build_image_index()
            self._save_indexes()

    def _check_saved_files(self):
        return (os.path.exists(os.path.join(self.save_dir, "text_embeddings.npy")) and
                os.path.exists(os.path.join(self.save_dir, "image_embeddings.npy")) and
                os.path.exists(os.path.join(self.save_dir, "text_index.faiss")) and
                os.path.exists(os.path.join(self.save_dir, "image_index.faiss")))

    def _save_indexes(self):
        np.save(os.path.join(self.save_dir, "text_embeddings.npy"), self.text_embeddings)
        np.save(os.path.join(self.save_dir, "image_embeddings.npy"), self.image_embeddings)
        faiss.write_index(self.text_index, os.path.join(self.save_dir, "text_index.faiss"))
        faiss.write_index(self.image_index, os.path.join(self.save_dir, "image_index.faiss"))
        with open(os.path.join(self.save_dir, "metadata.pkl"), "wb") as f:
            pickle.dump((self.captions, self.slides), f)
        print("✅ Saved embeddings, indexes, and metadata.")

    def _load_indexes(self):
        self.text_embeddings = np.load(os.path.join(self.save_dir, "text_embeddings.npy"))
        self.image_embeddings = np.load(os.path.join(self.save_dir, "image_embeddings.npy"))
        self.text_embeddings = self.text_embeddings / np.linalg.norm(self.text_embeddings, axis=1, keepdims=True)
        self.image_embeddings = self.image_embeddings / np.linalg.norm(self.image_embeddings, axis=1, keepdims=True)
        self.text_index = faiss.read_index(os.path.join(self.save_dir, "text_index.faiss"))
        self.image_index = faiss.read_index(os.path.join(self.save_dir, "image_index.faiss"))
        with open(os.path.join(self.save_dir, "metadata.pkl"), "rb") as f:
            self.captions, self.slides = pickle.load(f)
        print("✅ Loaded and normalized embeddings, indexes, and metadata.")

    def _load_dataset(self, image_dir: str, caption_dir: str) -> Tuple[List[Image.Image], List[str]]:
        slides = []
        captions = []
        speaker_folders = sorted(os.listdir(caption_dir))
        for speaker in speaker_folders:
            speaker_caption_dir = os.path.join(caption_dir, speaker)
            speaker_image_dir = os.path.join(image_dir, speaker)
            if not os.path.isdir(speaker_caption_dir) or not os.path.isdir(speaker_image_dir):
                continue
            caption_files = sorted([f for f in os.listdir(speaker_caption_dir) if f.endswith('.txt')])
            image_files = sorted([f for f in os.listdir(speaker_image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])
            for caption_file in caption_files:
                base_name = os.path.splitext(caption_file)[0]
                image_file = f"{base_name}.png"
                if image_file not in image_files:
                    image_file = f"{base_name}.jpg"
                    if image_file not in image_files:
                        continue
                cap_path = os.path.join(speaker_caption_dir, caption_file)
                img_path = os.path.join(speaker_image_dir, image_file)
                try:
                    image = Image.open(img_path).convert('RGB')
                    with open(cap_path, 'r', encoding='utf-8') as f:
                        caption = f.read().strip()
                    slides.append(image)
                    captions.append((speaker, caption, img_path))
                except Exception as e:
                    print(f"Error loading {img_path} or {cap_path}: {e}")
        print(f"Loaded {len(slides)} slide-caption pairs")
        return slides, captions

    def encode_text(self, texts: List[str]) -> np.ndarray:
        embeddings = self.text_encoder.encode(texts, normalize_embeddings=True)
        return np.array(embeddings)

    def _build_text_index(self):
        embeddings = self.text_encoder.encode([cap[1] for cap in self.captions], normalize_embeddings=True)
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(np.array(embeddings).astype('float32'))
        return index, embeddings

    def _build_image_index(self):
        ocr_texts = []
        for image in self.slides:
            result = self.ocr.readtext(np.array(image))
            text = "\n".join([line[1] for line in result])
            ocr_texts.append(text)
        embeddings = self.text_encoder.encode(ocr_texts, normalize_embeddings=True)
        embeddings = np.array(embeddings).astype('float32')
        dimension = embeddings.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(embeddings)
        return index, embeddings

    def retrieve(self, query: str, top_k: int, hybrid_weight: float, min_score_threshold: float) -> List[Dict]:
        """
        Retrieve relevant slide-caption pairs using hybrid retrieval.
        This version dynamically adjusts the hybrid_weight based on retrieval results.
        """
        # Reset hybrid_weight to 0 for each question
        #original_hybrid_weight = hybrid_weight
        retrievals = []

        # First retrieval attempt
        retrievals = self._perform_retrieval(query, top_k, hybrid_weight, min_score_threshold)
        # if len(retrievals) == 0:
        #     # If no slides retrieved, set hybrid_weight to 0.5 and try again
        #     hybrid_weight = 0.5
        #     print("🔵 No slides retrieved with hybrid_weight=0.0, trying hybrid_weight=0.5")
        #     retrievals = self._perform_retrieval(query, top_k, hybrid_weight, min_score_threshold)
            
        # if len(retrievals) == 0:
        #     # If still no slides retrieved, set hybrid_weight to 1 and try again
        #     print("🔵 No slides retrieved with hybrid_weight=0.5, trying hybrid_weight=1.0")
        #     hybrid_weight = 1.0
        #     retrievals = self._perform_retrieval(query, top_k, hybrid_weight, min_score_threshold)
        
        # Always revert hybrid_weight to its original value after processing
        #hybrid_weight = original_hybrid_weight
        print(f"🔍 Number of retrieved slides: {len(retrievals)}")
        total_slides = len(self.slides)
        #print(f"Total slides in lecture: {total_slides}")
        return retrievals, total_slides

    def _perform_retrieval(self, query: str, top_k: int, hybrid_weight: float, min_score_threshold: float) -> List[Dict]:
        """
        This function actually performs the retrieval based on the given hybrid_weight and other parameters.
        It's extracted to avoid repetition.
        """
        text_query_embedding = self.encode_text([query])[0].astype('float32')
        text_query_embedding = text_query_embedding / np.linalg.norm(text_query_embedding)

        # Text distances
        text_distances, text_indices = self.text_index.search(
            np.array([text_query_embedding]), top_k * 2
        )

        # OCR slide distances (uses same query)
        image_distances, image_indices = self.image_index.search(
            np.array([text_query_embedding]), top_k * 2
        )


        # Normalize text and image scores
        text_scores = [1 - (d / 2) for d in text_distances[0]]
        image_scores = [1 - (d / 2) for d in image_distances[0]]

        # Min-max normalization to scale scores to [0, 1]
        text_scores = (text_scores - np.min(text_scores)) / (np.max(text_scores) - np.min(text_scores) + 1e-9)
        image_scores = (image_scores - np.min(image_scores)) / (np.max(image_scores) - np.min(image_scores) + 1e-9)

        # Apply hybrid scoring
        # hybrid_scores = {}
        # for i, idx in enumerate(text_indices[0]):
        #     score = (1 - hybrid_weight) * text_scores[i]
        #     hybrid_scores[idx] = score

        # for i, idx in enumerate(image_indices[0]):
        #     score = hybrid_weight * image_scores[i]
        #     if idx in hybrid_scores:
        #         hybrid_scores[idx] += score
        #     else:
        #         hybrid_scores[idx] = score

        
        # Hybrid scores
        hybrid_scores = {}
        for i, idx in enumerate(text_indices[0]):
            # Convert L2 distance to cosine similarity
            cosine_similarity = 1 - (text_distances[0][i] / 2)
            score = (1 - hybrid_weight) * cosine_similarity  # Text contribution
            hybrid_scores[idx] = score

        for i, idx in enumerate(image_indices[0]):
            # Convert L2 distance to cosine similarity
            cosine_similarity = 1 - (image_distances[0][i] / 2)
            score = hybrid_weight * cosine_similarity  # Image contribution
            if idx in hybrid_scores:
                hybrid_scores[idx] += score
            else:
                hybrid_scores[idx] = score
        print(f"Hybrid scores: {hybrid_scores}")
        sorted_indices = sorted(hybrid_scores.keys(), key=lambda idx: hybrid_scores[idx], reverse=True)
        #print(hybrid_scores)
        # Filter main matches by min_score_threshold
        filtered_indices = [idx for idx in sorted_indices if hybrid_scores[idx] >= min_score_threshold][:top_k]
        main_indices = filtered_indices[:top_k]
        # Add neighbors ONLY if same speaker
        neighbor_indices = set()
        for idx in main_indices:
            main_speaker = self.captions[idx][0]
            for offset in [-2, -1, 1, 2]:
                neighbor_idx = idx + offset
                if 0 <= neighbor_idx < len(self.slides):
                    neighbor_speaker = self.captions[neighbor_idx][0]
                    if neighbor_speaker == main_speaker:  # Check same speaker
                        neighbor_indices.add(neighbor_idx)

        # Combine: main retrievals first, then neighbors (no duplicate)
        all_indices = list(main_indices) + list(neighbor_indices - set(main_indices))

        # Build final retrieval list
        # ...existing code...

# Build final retrieval list
        results = []
        for idx in all_indices:
            #print(self.captions)
            slide_type = "main" if idx in main_indices else "neighbor"
            results.append({
                'slide': self.slides[idx],
                'caption': self.captions[idx][1],
                'speaker': self.captions[idx][0],
                'slide_path': self.captions[idx][2],  # Add slide path here
                'score': hybrid_scores.get(idx, None),  # Neighbor slides may not have a score
                'type': slide_type  # Add the type here!
            })

        return results

        
    def evaluate_rag_accuracy(self, benchmark_path: str, output_path: str = "./benchmark/misclassified-rag.csv"):
        """
        Evaluate the RAG accuracy by checking if the true slide is retrieved.

        Args:
            benchmark_path: Path to the benchmark CSV file (e.g., mcq_labelled.csv).
            output_path: Path to save the misclassified results.
        """
        df = pd.read_csv(benchmark_path)
        misclassified = []
        total_questions = len(df)
        correct_retrievals = 0
        total_retrievals = 0

        for index, row in df.iterrows():
            question_id = row['Index']
            question = row['Question']
            true_slide_path = row['slide_path']
            speaker = row['Speaker']

            # Retrieve slides for the question
            retrievals, total_slides = self.retrieve(
                query=question,
                top_k=3,  # Adjust as needed
                hybrid_weight=0.5,
                min_score_threshold=0.3
            )
            #print(retrievals)
            total_retrievals += len(retrievals)
            # Extract retrieved slide paths and their types
            retrieved_slide_paths = [item['slide_path'] for item in retrievals]
            retrieved_slide_types = {item['slide_path']: item['type'] for item in retrievals}

            # Check if the true slide is in the retrieved slides
            if true_slide_path in retrieved_slide_paths:
                correct_retrievals += 1
                slide_type = retrieved_slide_types[true_slide_path]
            else:
                slide_type = "not_retrieved"

            # Record misclassifications
            if slide_type == "not_retrieved":
                misclassified.append({
                    "Index": question_id,
                    "Speaker": speaker,
                    "Question": question,
                    "True Slide": true_slide_path,
                    "Retrieved Slides": retrieved_slide_paths
                })

        # Calculate RAG accuracy
        rag_accuracy = correct_retrievals / total_questions if total_questions > 0 else 0.0
        print(f"✅ RAG Accuracy: {rag_accuracy:.2%} ({correct_retrievals}/{total_questions})")

        # Save misclassifications to a CSV file
        misclassified_df = pd.DataFrame(misclassified)
        misclassified_df.to_csv(output_path, index=False)
        print(f"✅ Misclassified results saved to {output_path}")
        print("Average number of retrievals per question:", total_retrievals / total_questions if total_questions > 0 else 0)
        print(f"Average number of slides: {total_slides}")


    def _load_llm(self):
        """Lazy-load the quantized VLM (Qwen2.5-VL) when needed."""
        if self.llm is None or self.tokenizer is None:
            print("🔵 Loading quantized VLM model (Qwen2.5-VL-AWQ)...")

            model_id = "Qwen/Qwen2.5-VL-3B-Instruct-AWQ"

            from awq import AutoAWQForCausalLM
            from transformers import AutoProcessor, AutoTokenizer

            self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
            self.processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True, use_fast=True)

            self.llm = AutoAWQForCausalLM.from_quantized(
                model_id,
                trust_remote_code=True,
                device_map="auto",
                safetensors=True  # or False, depends on your files
            )

            print("✅ Quantized VLM model loaded on GPU.")



    from qwen_vl_utils import process_vision_info

    def answer_question(
        self,
        query: str,
        a: str, b: str, c: str, d: str,
        system_prompt, suffix_prompt,
        top_k: int = 3,
        hybrid_weight: float = 0.5,
        min_score_threshold: float = 0.3
    ) -> str:
        """Answer a chemistry question using retrieved images and captions."""

        # Make sure the model is loaded
        self._load_llm()

        # Retrieve relevant slides
        retrievals, total_slides = self.retrieve(
            query, top_k=top_k, hybrid_weight=hybrid_weight, min_score_threshold=min_score_threshold
        )

        # Build image-caption pairs in Qwen message format
        image_content = []
        for i, item in enumerate(retrievals):
            #img = item['slide']
            img = self._resize_image(item['slide'], max_size=512)
            image_content.append({
                "type": "image",
                "image": img   # raw PIL.Image is fine, the processor will handle it
            })
            image_content.append({
                "type": "text",
                "text": f"Slide {i+1}: {item['caption']}"
            })

        # Build the full user message
        user_content = image_content + [
            {
                "type": "text",
                "text": (
                    f"\n\nQuestion:\n{query}\n"
                    f"a. {a}\n"
                    f"b. {b}\n"
                    f"c. {c}\n"
                    f"d. {d}\n"
                    f"\n{suffix_prompt}"
                )
            }
        ]

        messages = [
            {                "role": "system",
                "content": system_prompt},
            {
                "role": "user",
                "content": user_content
            }
        ]

        # Apply chat template
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        # Qwen-specific: build image tensors
        image_inputs, video_inputs = process_vision_info(messages)

        # Tokenize
        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to("cuda")

        print(f"Input text: {text}")
        assert len(text.strip()) > 0


        print(inputs)
        print(inputs.input_ids)
        print(inputs.input_ids.shape)

        # Generate
        with torch.no_grad():
            generated_ids = self.llm.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=True,
                temperature=0.7,  
                top_p=0.9,
                no_repeat_ngram_size=2,
                num_beams=1,
                pad_token_id=self.tokenizer.eos_token_id
            )


        # Only keep new tokens
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        output_text = self.processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0]

        return retrievals, output_text.strip()



    # def extract_answer(self, answer: str) -> str:
    #     # Look for patterns like: Answer: d or Answer:\n d
    #     match = re.search(r"Answer:\s*([a-dA-D])\b", answer)
    #     if match:
    #         return match.group(1).lower()
    #     else:
    #         print("⚠️ Invalid answer format:", answer[:200])  # Print only the first 200 chars for debugging
    #         return None

    # import re

    # def extract_answer(self, answer: str) -> str:
    #     match = re.search(r"Answer\s*[:\-]?\s*([a-dA-D])\b", answer)
    #     if match:
    #         return match.group(1).lower()
    #     else:
    #         print("⚠️ Could not extract answer:", answer[:200])
    #         return None

    import re

    def extract_answer(self, answer: str) -> str:
        # ✅ Try strict pattern first: Answer: a/b/c/d
        match = re.search(r"Answer\s*[:\-]?\s*([a-dA-D])\b", answer)
        if match:
            return match.group(1).lower()

        # ✅ Fallback: look for single letter at start of line
        lines = answer.strip().splitlines()
        if lines:
            first_line = lines[0].strip().lower()
            if first_line in {"a", "b", "c", "d"}:
                return first_line

        # ✅ Fallback: scan for lone a/b/c/d anywhere in first few lines
        fallback = re.search(r"\b([a-dA-D])\b", "\n".join(lines[:2]))
        if fallback:
            return fallback.group(1).lower()

        print(f"⚠️ Could not extract answer:\n{answer[:200]}")
        return None

        
    def choose_slide(self, retrievals, query):
        """Answer a chemistry question using retrieved images and captions."""

        images = []

        for i, item in enumerate(retrievals):
            images.append(item['slide'])
            prompt += f"\nSlide {i+1}: {item['caption']} <img>"

        assert len(images) == prompt.count("<img>"), f"Image-token mismatch: {len(images)} images vs {prompt.count('<img>')} tokens"


        prompt += "Given the following question, choose the (image, caption) pair that you think the question is about. Choose only one pair and return the index of the pair in the format 'Slide X'. Explain your reasoning.\n"
        prompt += (
            f"\n\nQuestion:\n{query}\n"
        )

        
        
        inputs = self.processor(text=prompt, images=images, return_tensors="pt").to(self.llm.device)

        with torch.no_grad():
            output = self.llm.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                temperature=0.0,
                top_p=0.7,
                no_repeat_ngram_size=2,
                early_stopping=True
            )

        answer = self.processor.batch_decode(output, skip_special_tokens=True)[0]
        return retrievals, answer.strip()
            
    # ✅ Now your heavy work is only triggered when you run:
    # rag = ChemistryRAG(...)
    # rag.build()   <-- only once!
    # rag.answer_question(...)   <-- fast tweaks after

    from PIL import Image

    def _resize_image(self, image: Image.Image, max_size: int = 512) -> Image.Image:
        """
        Resize the image so that its longest side is `max_size` pixels.
        Keeps aspect ratio.
        """
        w, h = image.size
        scale = max_size / max(w, h)
        if scale >= 1:
            return image  # already small enough
        new_w = int(w * scale)
        new_h = int(h * scale)
        return image.resize((new_w, new_h), Image.LANCZOS)


In [ ]:

# Initialize the RAG system with the specified paths
rag = ChemistryRAG(
    train_caption_dir="data/4m_data/english/train/caption",
    train_image_dir="data/4m_data/english/train/rgb",
    test_caption_dir="data/4m_data/english/test/caption",
    test_image_dir="data/4m_data/english/test/rgb",
    save_dir="./saved_indexes",
    load_saved=True # ⬅️ # False to rebuild indexes
)

In [ ]:
rag.build()

In [ ]:
#When you change a method in ChemistryRAG, you can re-run the cell below to update the rag instance.
# This is useful if you modify the class definition and want to keep the same instance.
rag.__class__ = ChemistryRAG

In [ ]:
#Test the RAG system: get accuracy
acc = rag.evaluate_rag_accuracy(benchmark_path="benchmark/data/mcq_labelled.csv",)


In [ ]:
# system_prompt= (
#             "You are an expert chemistry tutor. Based only on the following image-caption pairs, answer the multiple-choice question below.\n" \
#             "The question is about one of the image-caption pairs. Start by finding the single pair the question is asking about. Then think through the image-caption context carefully and explain your reasoning. Then, based on your reasoning, choose the correct answer between options a, b, c, or d. There is only one correct answer.\n" \
#             "Your answer should have the following format:\n" \
#             "Answer: a, b, c, or d.\n" \
#             "Explanation: <your reasoning here>\n" \
#             "Do not include any other text.\n\n"
            
#         )

# system_prompt = ( "You MUST output your final answer in the exact format:\n"\
# "Answer: <letter>\n"\
# "Explanation: <your explanation here>\n\n"\
# "Do not put anything on the 'Answer:' line except one letter (a/b/c/d). \n"\
# "Do not repeat the question.\n")

#0.4
system_prompt= ("You MUST first list the pros and cons for each option (a, b, c, d).\n"\
"Then choose ONE option and output the final answer in EXACT format:\n"
"Answer: <letter>\n"
"Explanation: <your reasoning>\n"
"Only the final answer must follow the format.\n"
)

#0.3
# system_prompt = ("You MUST first choose the SINGLE most relevant image to answer the question. Once you have chosen one image, you must ignore the other images. Then you MUST list the pros and cons for each option (a, b, c, d).\n"\
# "Then choose ONE option and output the final answer in EXACT format:\n"
# "Answer: <letter>\n"
# "Explanation: <your reasoning>\n"
# "Only the final answer must follow the format.\n"
# )

#0.35
# system_prompt = (
#     "You MUST first look at all the given images and choose the ONE most relevant image for answering the question.\n"
#     "After choosing, completely ignore the other images.\n\n"
#     "Next, analyze the question carefully and list the pros and cons for each answer option (a, b, c, d).\n\n"
#     "Then choose ONE final answer and output it in EXACTLY this format (no extra text):\n\n"
#     "Answer: <letter>\n"
#     "Explanation: <your detailed reasoning here>\n\n"
#     "Do not include anything else after the answer."
# )


suffix_prompt = ("There is only one correct answer.\n"\
"Strictly follow the format: Answer: <letter>. Then on the next line, Explanation: ...\n"\
"Do not include any other text.\n\n")


In [ ]:
count = 20
df = pd.read_csv('benchmark/data/mcq_labelled.csv')
results = []
correct = 0
for index, row in df.head(count).iterrows():
    # Extract question and other details
    question = row['Question']
    a = row['a']
    b = row['b']
    c = row['c']
    d = row['d']
    options = [a, b, c, d]
    true_label = row['Answer']


    retrievals, final_answer = rag.answer_question(question, a, b, c, d, system_prompt, suffix_prompt, top_k=3, hybrid_weight=0.5, min_score_threshold=0.3)
#   retrievals = rag.retrieve(query=question)
    print(final_answer)
    generated_answer = rag.extract_answer(final_answer)
    if generated_answer not in {"a", "b", "c", "d"}:
        print(f"⚠️ Invalid answer format: {generated_answer}")
        count -= 1
        continue  # skip invalid predictions
    elif generated_answer == true_label:
        correct += 1

accuracy = correct / count
print(accuracy)

Prompt chaining: 
1. Ask the model to find the slide
2. Ask the model to answer the question

In [ ]:
def choose_slide(rag, retrievals, query):
    """Answer a chemistry question using retrieved images and captions."""

    images = []

    for i, item in enumerate(retrievals):
        images.append(item['slide'])
        prompt += f"\nSlide {i+1}: {item['caption']} <|image|>"

    prompt += "Given the following question, choose the (image, caption) pair that you think the question is about. Choose only one pair and return the index of the pair in the format 'Slide X'. Explain your reasoning.\n"
    prompt += (
        f"\n\nQuestion:\n{query}\n"
    )

    
    
    inputs = rag.processor(text=prompt, images=images, return_tensors="pt").to(rag.llm.device)

    with torch.no_grad():
        output = rag.llm.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            temperature=0.9,
            top_p=0.9,
            no_repeat_ngram_size=2,
            early_stopping=True
        )

    answer = rag.processor.batch_decode(output, skip_special_tokens=True)[0]
    return retrievals, answer.strip()

In [ ]:
for index, row in df.iloc[:20].iterrows():
    # Extract question and other details
    question = row['Question']
    a = row['a']
    b = row['b']
    c = row['c']
    d = row['d']
    options = [a, b, c, d]
    true_label = row['Answer']
    chosen_slide = choose_slide(rag,retrievals, question)
    print(chosen_slide)

In [ ]:
count = 54
df = pd.read_csv('benchmark/data/mcq_labelled.csv')
results = []
correct = 0
for index, row in df.iloc[:20].iterrows():
    # Extract question and other details
    question = row['Question']
    a = row['a']
    b = row['b']
    c = row['c']
    d = row['d']
    options = [a, b, c, d]
    true_label = row['Answer']

 
    retrievals, final_answer = rag.answer_question(question, a, b, c, d, top_k=6, hybrid_weight=0.5, min_score_threshold=0.3)
#   retrievals = rag.retrieve(query=question)
    print(final_answer)
    generated_answer = rag.extract_answer(final_answer)
    if generated_answer not in {"a", "b", "c", "d"}:
        print(f"⚠️ Invalid answer format: {generated_answer}")
        count -= 1
        continue  # skip invalid predictions
    elif generated_answer == true_label:
        correct += 1

accuracy = correct / count
print(accuracy)